In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt) 
    add_assistant_message(messages, "```json") #prefilling the answer, guiding claude to generate the response in the desired format
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)




In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [7]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [8]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extractor from S3 Bucket URL\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_aws_region(s3_url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        s3_url (str): S3 bucket URL in format 's3://bucket-name.region.amazonaws.com'\n        \n    Returns:\n        str: The AWS region (e.g., 'us-east-1'), or None if not found\n        \n    Raises:\n        ValueError: If the URL format is invalid\n    \"\"\"\n    if not s3_url:\n        raise ValueError(\"S3 URL cannot be empty\")\n    \n    # Pattern to match: s3://bucket-name.region.amazonaws.com\n    pattern = r's3://[a-z0-9.-]+\\.([a-z0-9-]+)\\.amazonaws\\.com'\n    \n    match = re.search(pattern, s3_url)\n    \n    if match:\n        return match.group(1)\n    else:\n        raise ValueError(f\"Invalid S3 URL format: {s3_url}\")\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Test 1: Sta